In [7]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


import sys
sys.path.append('../')
import sionna

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible random number generation

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import *
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, BinarySource
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper

In [8]:
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
print('Number of GPUs available :', len(gpus))
if gpus:
    gpu_num = 0 # Index of the GPU to be used
    try:
        #tf.config.set_visible_devices([], 'GPU')
        tf.config.set_visible_devices(gpus[gpu_num], 'GPU')
        print('Only GPU number', gpu_num, 'used.')
        tf.config.experimental.set_memory_growth(gpus[gpu_num], True)
    except RuntimeError as e:
        print(e)

Number of GPUs available : 0


In [9]:
from tensorflow.keras.layers import Layer, Conv2D, LayerNormalization, SeparableConv2D
from tensorflow.nn import relu
class ResidualBlock(tf.keras.Model):
    r"""
    This Keras layer implements a convolutional residual block made of two convolutional layers with ReLU activation, layer normalization, and a skip connection.
    The number of convolutional channels of the input must match the number of kernel of the convolutional layers ``num_conv_channel`` for the skip connection to work.

    Input
    ------
    : [batch size, num time samples, num subcarriers, num_conv_channel], tf.float
        Input of the layer

    Output
    -------
    : [batch size, num time samples, num subcarriers, num_conv_channel], tf.float
        Output of the layer
    """

    def build(self, input_shape):

        # Layer normalization is done over the last three dimensions: time, frequency, conv 'channels'
        self._layer_norm_1 = LayerNormalization(axis=(-1, -2, -3))
        self._conv_1 = SeparableConv2D(filters= 64,
                              kernel_size=[3,3],
                              padding='same',
                              activation=None)
        # Layer normalization is done over the last three dimensions: time, frequency, conv 'channels'
        self._layer_norm_2 = LayerNormalization(axis=(-1, -2, -3))
        self._conv_2 = SeparableConv2D(filters= 128,
                              kernel_size=[3,3],
                              padding='same',
                              activation=None)

    def call(self, inputs):
        z = self._layer_norm_1(inputs)
        z = relu(z)
        z = self._conv_1(z)
        z = self._layer_norm_2(z)
        z = relu(z)
        z = self._conv_2(z) # [batch size, num time samples, num subcarriers, num_channels]
        # Skip connection
        z = z + inputs

        return z

class CustomNeuralReceiver(tf.keras.Model):
    r"""
    Keras layer implementing a residual convolutional neural receiver.

    This neural receiver is fed with the post-DFT received samples, forming a resource grid of size num_of_symbols x fft_size, and computes LLRs on the transmitted coded bits.
    These LLRs can then be fed to an outer decoder to reconstruct the information bits.

    Input
    ------
    y_no: [batch size, num ofdm symbols, num subcarriers, 2*num rx antenna + 1], tf.float32
        Concatenated received samples and noise variance.
(
    y : [batch size, num rx antenna, num ofdm symbols, num subcarriers], tf.complex
        Received post-DFT samples.

    no : [batch size], tf.float32
        Noise variance. At training, a different noise variance value is sampled for each batch example.
)
    Output
    -------
    : [batch size, num ofdm symbols, num subcarriers, num_bits_per_symbol]
        LLRs on the transmitted bits.
    """

    def __init__(self, training = False):
        super(CustomNeuralReceiver, self).__init__()
        self._training = training

    def build(self, input_shape):

        # Input convolution
        self._input_conv = Conv2D(filters= 128,
                                  kernel_size=[3,3],
                                  padding='same',
                                  activation=None)
        # Residual blocks
        self._res_block_1 = ResidualBlock()
        self._res_block_2 = ResidualBlock()
        self._res_block_3 = ResidualBlock()
        self._res_block_4 = ResidualBlock()
        # Output conv
        self._output_conv = Conv2D(filters= 2,    # QPSK
                                   kernel_size=[3,3],
                                   padding='same',
                                   activation=None)
        

    @tf.function(jit_compile=True)
    def call(self, inputs):
        # Input conv
        z = self._input_conv(inputs)
        # Residual blocks
        z = self._res_block_1(z)
        z = self._res_block_2(z)
        z = self._res_block_3(z)
        z = self._res_block_4(z)
        # Output conv
        z = self._output_conv(z)
        # if self._training == False:
        #     z = tf.cast(z * (2**7), tf.int8)
        return z
    

import pickle
_model = CustomNeuralReceiver(training = False)
inputs = tf.zeros([1,3276,14,16])
_model(inputs)
_model.summary()

def load_weights(model, pretrained_weights_path):
    # Build Model with random input
    # Load weights
  with open(pretrained_weights_path, 'rb') as f:
    weights = pickle.load(f)
    model.set_weights(weights)
    print(f"Loaded pretrained weights from {pretrained_weights_path}")

#load_weights(_model, '/content/drive/MyDrive/Pusch_data/Model_weights/model_weight_FULL_RB_epoch_40.pkl')
load_weights(_model, '../model_weight_FULL_RB_epoch_40.pkl')


Model: "custom_neural_receiver_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_2 (Conv2D)           multiple                  18560     
                                                                 
 residual_block_4 (Residual  multiple                  17630080  
 Block)                                                          
                                                                 
 residual_block_5 (Residual  multiple                  17630080  
 Block)                                                          
                                                                 
 residual_block_6 (Residual  multiple                  17630080  
 Block)                                                          
                                                                 
 residual_block_7 (Residual  multiple                  17630080  
 Block)                                   

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class SystemConfig:
    NCellId: int = 246
    FrequencyRange: int = 1
    BandWidth: int = 100
    Numerology: int = 1
    CpType: int = 0
    NTxAnt: int = 1
    NRxAnt: int = 8
    BwpNRb: int = 273
    BwpRbOffset: int = 0
    harqProcFlag: int = 0
    nHarqProc: int = 1
    rvSeq: int = 0


@dataclass
class UeConfig:
    rvIdx: int = 0
    TransformPrecoding: int = 0
    Rnti: int = 20002
    nId: int = 246
    CodeBookBased: int = 0
    DmrsPortSetIdx: List[int] = field(default_factory=lambda: [0])  # FIXED
    NLayers: int = 1
    NumDmrsCdmGroupsWithoutData: int = 2
    Tpmi: int = 0
    FirstSymb: int = 0
    NPuschSymbAll: int = 14
    RaType: int = 1
    FirstPrb: int = 31
    NPrb: int = 4
    FrequencyHoppingMode: int = 0
    McsTable: int = 0
    Mcs: int = 3
    ILbrm: int = 0
    nScId: int = 0
    NnScIdId: int = 246
    DmrsConfigurationType: int = 0
    DmrsDuration: int = 1
    DmrsAdditionalPosition: int = 1
    PuschMappingType: int = 0
    DmrsTypeAPosition: int = 3
    HoppingMode: int = 0
    NRsId: int = 0
    Ptrs: int = 0
    ScalingFactor: int = 0
    OAck: int = 0
    IHarqAckOffset: int = 11
    OCsi1: int = 0
    ICsi1Offset: int = 7
    OCsi2: int = 0
    ICsi2Offset: int = 0
    NPrbOh: int = 0
    nCw: int = 1
    TpPi2Bpsk: int = 0

@dataclass
class MyConfig:
    Sys: SystemConfig
    Ue: List[UeConfig]
    Num_tx: int = 1
    Num_rx: int = 1
    Carrier_frequency: float = 2.55e9  # Carrier frequency in Hz

# Example usage
My_Config = MyConfig(SystemConfig(), [UeConfig()])

In [11]:
import re
import math

def bitmask_to_indices(bitmask):
    indices = []
    index = 0
    while bitmask:
        if bitmask & 1:
            indices.append(index)
        bitmask >>= 1
        index += 1
    return indices

def config_parser(config_path):
    caseInfo = {}
    sysInfo = {}
    ue = {}
    chcfg = {}
    auxInfo = {}
    with open(config_path, 'r') as file:
        for num, line in enumerate(file, 1):
            line = line.strip()
            if line and not line.startswith('%'):  # Ignore empty or comment lines
                #read case information and store it in caseInfo
                key, value = line.split('=')
                key = key.strip()
                value = value.strip('; ').strip()
                if value.lower() == 'true': 
                    value = True
                elif value.lower() == 'false':
                    value = False
                elif value.isdigit():  # Convert to integer if the value is a number
                    value = int(value)
                if num < 3:
                    caseInfo[key] = value    
                else:
                    #read cell information  
                    if key.startswith('sys'):
                        _, value2 = key.split('.')
                        sysInfo[value2] = value
                    #read chcfg information
                    elif key.startswith('chcfg'): 
                        _, value2 = key.split('.')
                        chcfg[value2] = value
                    #read ue config
                    elif key.startswith('ue'): 
                        key2, value2 = key.split('.')
                        ue_idx = re.search(r"\{([^}]+)\}", key2)
                        ue_idx = ue_idx.group(1)
                        if ue_idx.isdigit():
                            ue_idx = int(ue_idx)
                        ue_idx = ue_idx - 1
                        if is_empty(ue, ue_idx) == 0 :
                            #create an empty config dictionary for ue_idx                     
                            ue[ue_idx] = {}
                        if value2 == 'DmrsPortSetIdx':
                            ue[ue_idx][value2] = bitmask_to_indices(value)
                        else:
                            ue[ue_idx][value2] = value
                    else:
                        auxInfo[key] = value                
    return caseInfo, sysInfo, ue, chcfg, auxInfo    

def is_empty(dictionary, key):
    # Check if the key exists and if the value is considered "empty"
    if key in dictionary:
        return True
    return False

def fft_size_return(n):
    if n <= 1:
        return 1    
    if n >= 0.85*2**math.ceil(math.log2(n)):
        return 2**(math.ceil(math.log2(n))+1)
    else:
        return 2 ** math.ceil(math.log2(n))

In [12]:
from pathlib import Path
import struct
import numpy as np

# Path to the data file
field_dir = '../Pusch_data/data_field'
data_path = f'{field_dir}/dump_pass_sfn_6_sf_5_freq.bin'
cfg_path = f'{field_dir}/dump_pass_sfn_6_sf_5_cfg.txt'

freq = []

with open(data_path, 'rb') as file:
    binary_data = file.read()
    
    # Loop through the binary data in chunks of 2 bytes (since int16 is 2 bytes)
    for i in range(0, len(binary_data), 4):
        # Convert the chunk to an integer and append it to the list
        real = binary_data[i:i+2]
        imag = binary_data[i+2:i+4]

        #unpack the 2 bytes into a little-endian int16
        if len(real) == 2:
            real_part = struct.unpack('<h', real)[0]
            imag_part = struct.unpack('<h', imag)[0]
            freq.append(complex(real_part, imag_part))

freq = np.array(freq)
print(freq.shape)
# print("Unpacked int16 value:\n", freq)

# Load configuration parameters from the file .txt
caseInfo, sysInfo, ueInfo, chcfg, auxInfo = config_parser(cfg_path)  

#reshape the iq data into resource grid (Numsubcarrier x 14 Sym x 8 ant)
freq = freq.reshape((sysInfo['BwpNRb']*12, 14, sysInfo['NRxAnt'], ))
print(freq.shape)
iqUe = freq[ueInfo[0]['FirstPrb']*12:(ueInfo[0]['NPrb'] + ueInfo[0]['FirstPrb'] + 1)*12,:, :]
iqUe = iqUe[np.newaxis,:]
iqUe = iqUe.astype(np.complex64)

(217728,)
(1944, 14, 8)


In [13]:
System_Config = SystemConfig(**sysInfo)

In [14]:
Ue_Configs = [UeConfig(**ue) for ue in ueInfo.values()]

In [15]:
My_Config = MyConfig(System_Config, Ue_Configs)
My_Config

MyConfig(Sys=SystemConfig(NCellId=246, FrequencyRange=1, BandWidth=60, Numerology=1, CpType=0, NTxAnt=1, NRxAnt=8, BwpNRb=162, BwpRbOffset=0, harqProcFlag=0, nHarqProc=1, rvSeq=0), Ue=[UeConfig(rvIdx=0, TransformPrecoding=0, Rnti=20002, nId=246, CodeBookBased=0, DmrsPortSetIdx=[0], NLayers=1, NumDmrsCdmGroupsWithoutData=2, Tpmi=0, FirstSymb=0, NPuschSymbAll=14, RaType=1, FirstPrb=22, NPrb=140, FrequencyHoppingMode=0, McsTable=0, Mcs=7, ILbrm=0, nScId=0, NnScIdId=246, DmrsConfigurationType=0, DmrsDuration=1, DmrsAdditionalPosition=2, PuschMappingType=0, DmrsTypeAPosition=3, HoppingMode=0, NRsId=0, Ptrs=0, ScalingFactor=0, OAck=0, IHarqAckOffset=11, OCsi1=0, ICsi1Offset=0, OCsi2=0, ICsi2Offset=0, NPrbOh=0, nCw=1, TpPi2Bpsk=0)], Num_tx=1, Num_rx=1, Carrier_frequency=2550000000.0)

In [16]:
class MyPUSCHConfig(PUSCHConfig):
    def __init__(self, My_Config: MyConfig):
        self.My_Config = My_Config
        super().__init__(
            carrier_config=CarrierConfig(
                n_cell_id=My_Config.Sys.NCellId,
                cyclic_prefix="normal" if ~My_Config.Sys.CpType else "extended",
                subcarrier_spacing=15*(2**My_Config.Sys.Numerology),
                n_size_grid=My_Config.Sys.BwpNRb,
                n_start_grid=My_Config.Sys.BwpRbOffset,
                slot_number=4,
                frame_number=0
            ),
            pusch_dmrs_config=PUSCHDMRSConfig(
                config_type=My_Config.Ue[0].DmrsConfigurationType + 1,
                length=My_Config.Ue[0].DmrsDuration,
                additional_position=My_Config.Ue[0].DmrsAdditionalPosition,
                dmrs_port_set=My_Config.Ue[0].DmrsPortSetIdx,
                n_id=My_Config.Ue[0].NnScIdId,
                n_scid=My_Config.Ue[0].nScId,
                num_cdm_groups_without_data=My_Config.Ue[0].NumDmrsCdmGroupsWithoutData,
                type_a_position=My_Config.Ue[0].DmrsTypeAPosition
            ),
            tb_config=TBConfig(
                channel_type='PUSCH',
                n_id=My_Config.Ue[0].nId,
                mcs_table=My_Config.Ue[0].McsTable + 1,
                mcs_index=My_Config.Ue[0].Mcs
            ),
            mapping_type='A' if ~My_Config.Ue[0].PuschMappingType else 'B',
            n_size_bwp=My_Config.Sys.BwpNRb,
            n_start_bwp=My_Config.Sys.BwpRbOffset,
            num_layers=My_Config.Ue[0].NLayers,
            num_antenna_ports=len(My_Config.Ue[0].DmrsPortSetIdx),
            precoding='non-codebook' if ~My_Config.Ue[0].CodeBookBased else 'codebook',
            tpmi=My_Config.Ue[0].Tpmi,
            transform_precoding=False if ~My_Config.Ue[0].TransformPrecoding else True,
            n_rnti=My_Config.Ue[0].Rnti,
            symbol_allocation=[My_Config.Ue[0].FirstSymb,My_Config.Ue[0].NPuschSymbAll]
        )
    @property
    def first_resource_block(self):
        """
        :class:`~sionna.nr.CarrierConfig` : Carrier configuration
        """
        return self.My_Config.Ue[0].FirstPrb
    
    @property
    def first_subcarrier(self):
        """
        :class:`~sionna.nr.CarrierConfig` : Carrier configuration
        """
        return 12*self.first_resource_block
    
    @property
    def num_resource_blocks(self):
        """
        int, read-only : Number of allocated resource blocks for the
            PUSCH transmissions.
        """
        return self.My_Config.Ue[0].NPrb

    @property
    def dmrs_grid(self):
        # pylint: disable=line-too-long
        """
        complex, [num_dmrs_ports, num_subcarriers, num_symbols_per_slot], read-only : Empty
            resource grid for each DMRS port, filled with DMRS signals

            This property returns for each configured DMRS port an empty
            resource grid filled with DMRS signals as defined in
            Section 6.4.1.1 [3GPP38211]. Not all possible options are implemented,
            e.g., frequency hopping and transform precoding are not available.

            This property provides the *unprecoded* DMRS for each configured DMRS port.
            Precoding might be applied to map the DMRS to the antenna ports. However,
            in this case, the number of DMRS ports cannot be larger than the number of
            layers.
        """
        # Check configuration
        self.check_config()

        # Configure DMRS ports set if it has not been set
        reset_dmrs_port_set = False
        if len(self.dmrs.dmrs_port_set)==0:
            self.dmrs.dmrs_port_set = list(range(self.num_layers))
            reset_dmrs_port_set = True

        # Generate empty resource grid for each port
        a_tilde = np.zeros([len(self.dmrs.dmrs_port_set),
                            self.num_subcarriers,
                            self.carrier.num_symbols_per_slot],
                            dtype=complex)
        first_subcarrier = self.first_subcarrier
        num_subcarriers = self.num_subcarriers

        # For every l_bar
        for l_bar in self.l_bar:

            # For every l_prime
            for l_prime in self.l_prime:

                # Compute c_init
                l = l_bar + l_prime
                c_init = self.c_init(l)
                # Generate RNG
                c = generate_prng_seq(first_subcarrier + num_subcarriers, c_init=c_init)
                c = c[first_subcarrier:]

                # Map to QAM
                r = 1/np.sqrt(2)*((1-2*c[::2]) + 1j*(1-2*c[1::2]))

                # For every port in the dmrs port set
                for j_ind, _ in enumerate(self.dmrs.dmrs_port_set):

                    # For every n
                    for n in self.n:

                        # For every k_prime
                        for k_prime in [0, 1]:

                            if self.dmrs.config_type==1:
                                k = 4*n + 2*k_prime + \
                                    self.dmrs.deltas[j_ind]
                            else: # config_type == 2
                                k = 6*n + k_prime + \
                                    self.dmrs.deltas[j_ind]

                            a_tilde[j_ind, k, self.l_ref+l] = \
                                r[2*n + k_prime] * \
                                self.dmrs.w_f[k_prime][j_ind] * \
                                self.dmrs.w_t[l_prime][j_ind]

        # Amplitude scaling
        a = self.dmrs.beta*a_tilde

        # Reset DMRS port set if it was not set
        if reset_dmrs_port_set:
            self.dmrs.dmrs_port_set = []

        return a
    
Pusch_Config = MyPUSCHConfig(My_Config)
Pusch_Config.show()

Carrier Configuration
cyclic_prefix : normal
cyclic_prefix_length : 2.3437500000000002e-06
frame_duration : 0.01
frame_number : 0
kappa : 64.0
mu : 1
n_cell_id : 246
n_size_grid : 162
n_start_grid : 0
num_slots_per_frame : 20
num_slots_per_subframe : 2
num_symbols_per_slot : 14
slot_number : 4
sub_frame_duration : 0.001
subcarrier_spacing : 30
t_c : 5.086263020833334e-10
t_s : 3.2552083333333335e-08

PUSCH Configuration
My_Config : MyConfig(Sys=SystemConfig(NCellId=246, FrequencyRange=1, BandWidth=60, Numerology=1, CpType=0, NTxAnt=1, NRxAnt=8, BwpNRb=162, BwpRbOffset=0, harqProcFlag=0, nHarqProc=1, rvSeq=0), Ue=[UeConfig(rvIdx=0, TransformPrecoding=0, Rnti=20002, nId=246, CodeBookBased=0, DmrsPortSetIdx=[0], NLayers=1, NumDmrsCdmGroupsWithoutData=2, Tpmi=0, FirstSymb=0, NPuschSymbAll=14, RaType=1, FirstPrb=22, NPrb=140, FrequencyHoppingMode=0, McsTable=0, Mcs=7, ILbrm=0, nScId=0, NnScIdId=246, DmrsConfigurationType=0, DmrsDuration=1, DmrsAdditionalPosition=2, PuschMappingType=0, DmrsT

In [128]:
class MySimulator():
    def __init__(self, pusch_config: MyPUSCHConfig):

        self.Num_rx = pusch_config.My_Config.Num_rx
        self.Num_tx = pusch_config.My_Config.Num_tx
    
        tb_size = pusch_config.tb_size
        num_coded_bits = pusch_config.num_coded_bits
        target_coderate = pusch_config.tb.target_coderate
        num_bits_per_symbol = pusch_config.tb.num_bits_per_symbol

        num_layers = pusch_config.num_layers
        n_rnti = pusch_config.n_rnti
        n_id = pusch_config.tb.n_id

        self.Binary_Source = BinarySource(dtype=tf.float32)
        self.TB_Encoder = TBEncoder(target_tb_size=tb_size,
                            num_coded_bits=num_coded_bits,
                            target_coderate=target_coderate,
                            num_bits_per_symbol=num_bits_per_symbol,
                            num_layers=num_layers,
                            n_rnti=n_rnti,
                            n_id=n_id,
                            channel_type="PUSCH",
                            codeword_index=0,
                            use_scrambler=True,
                            verbose=False,
                            output_dtype=tf.float32)
        
        self.Constellation_Mapper = Mapper("qam", num_bits_per_symbol, dtype=tf.complex64)

        self.Layer_Mapper = LayerMapper(num_layers=num_layers, dtype=tf.complex64)
    
        self.Pilot_Pattern = PUSCHPilotPattern([pusch_config], dtype=tf.complex64)

        num_subcarriers = pusch_config.num_subcarriers
        subcarrier_spacing = pusch_config.carrier.subcarrier_spacing*1e3
        fft_size = num_subcarriers
        cp_length = 48
        guard_subcarriers = (0,0)
        # Define the resource grid.
        resource_grid = ResourceGrid(
            num_ofdm_symbols=14,
            fft_size=fft_size,
            subcarrier_spacing=subcarrier_spacing,
            num_tx=self.Num_tx,
            num_streams_per_tx=1,
            cyclic_prefix_length=cp_length,
            num_guard_carriers=guard_subcarriers,
            dc_null=False,
            pilot_pattern=self.Pilot_Pattern,
            dtype=tf.complex64
        )

        self.Resource_Grid_Mapper = ResourceGridMapper(resource_grid, dtype=tf.complex64)        
        
        self.AWGN = AWGN()

 
        self.Channel_Estimator = PUSCHLSChannelEstimator(
                        resource_grid,
                        pusch_config.dmrs.length,
                        pusch_config.dmrs.additional_position,
                        pusch_config.dmrs.num_cdm_groups_without_data,
                        interpolation_type='nn',
                        dtype=tf.complex64)

        rxtx_association = np.ones([self.Num_rx, self.Num_tx], bool)
        stream_management = StreamManagement(rxtx_association, pusch_config.num_layers)
        self.Mimo_Detector = LinearDetector("lmmse", "bit", "maxlog", resource_grid, stream_management,
                                    "qam", pusch_config.tb.num_bits_per_symbol, dtype=tf.complex64)
        

        self.Layer_Demapper = LayerDemapper(self.Layer_Mapper, num_bits_per_symbol=num_bits_per_symbol)
        self.TB_Decode = TBDecoder(self.TB_Encoder, output_dtype=tf.float32)

        self.tb_size = tb_size
        self.resource_grid = resource_grid
        self.pusch_config = pusch_config
        
    def update_pilots(self, pilots):
        self.Resource_Grid_Mapper._resource_grid.pilot_pattern.pilots = pilots

        self.Channel_Estimator = PUSCHLSChannelEstimator(
                        self.Resource_Grid_Mapper._resource_grid,
                        self.pusch_config.dmrs.length,
                        self.pusch_config.dmrs.additional_position,
                        self.pusch_config.dmrs.num_cdm_groups_without_data,
                        interpolation_type='nn',
                        dtype=tf.complex64)

        rxtx_association = np.ones([self.Num_rx, self.Num_tx], bool)
        stream_management = StreamManagement(rxtx_association, self.pusch_config.num_layers)
        self.Mimo_Detector = LinearDetector("lmmse", "bit", "maxlog", self.Resource_Grid_Mapper._resource_grid, stream_management,
                                    "qam", self.pusch_config.tb.num_bits_per_symbol, dtype=tf.complex64)

    def sim(self, batch_size, channel_model, no_scaling, from_binary_source=True, gen_seed=2004*10+4):
        if from_binary_source:
            b = self.Binary_Source([batch_size, self.Num_tx, self.tb_size])
        else:
            b = tf.reshape(tf.constant(generate_prng_seq(batch_size * self.Num_tx * self.tb_size, gen_seed), dtype=tf.float32), [batch_size, self.Num_tx, self.tb_size])
        c = self.TB_Encoder(b)
        x_map = self.Constellation_Mapper(c)
        x_layer = self.Layer_Mapper(x_map)
        x = self.Resource_Grid_Mapper(x_layer)

        y, h = channel_model(x)
        no = no_scaling * tf.math.reduce_variance(y)

        y = self.AWGN([y, no])

        # if self.Rx_included:
        #     no_ = 0.001
        #     h_hat,err_var = self.Channel_Estimator([y, no_])
        #     llr_det = self.Mimo_Detector([y, h_hat, err_var, no_])
        #     llr_layer = self.Layer_Demapper(llr_det)

        #     b_hat, tb_crc_status = self.TB_Decode(llr_layer)

        #     return tb_crc_status, b, c, y
        
        return b, c, y
    
    def rec(self, y):
        no_ = 0.001
        h_hat, err_var = self.Channel_Estimator([y, no_])
        llr_det = self.Mimo_Detector([y, h_hat, err_var, no_])
        llr_layer = self.Layer_Demapper(llr_det)
        b_hat, tb_crc_status = self.TB_Decode(llr_layer)

        return b_hat, llr_det, tb_crc_status
        
ue_antenna = Antenna(polarization="single",
                polarization_type="V",
                antenna_pattern="38.901",
                carrier_frequency=2.55e9)

gnb_array = AntennaArray(num_rows=1,
                        num_cols=8//2,
                        polarization="dual",
                        polarization_type="cross",
                        antenna_pattern="38.901",
                        carrier_frequency=2.55e9)

channel_model = CDL(model = 'C',
                            delay_spread = 150*1e-9,
                            carrier_frequency = 2.55e9,
                            ut_array = ue_antenna,
                            bs_array = gnb_array,
                            direction = 'uplink',
                            min_speed = 1,
                            max_speed = 1)

In [129]:
simulator = MySimulator(Pusch_Config)

channel = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid, 
                                    add_awgn=False, normalize_channel=True, return_channel=True)

In [130]:
b, c, y = simulator.sim(1, channel, 0.1, False, 20044)

In [131]:
b_hat, c_soft, crc = simulator.rec(y)
print(crc)

tf.Tensor([[ True]], shape=(1, 1), dtype=bool)


In [132]:
np.abs(y.numpy()).max(), y.shape

(2.820779, TensorShape([1, 1, 8, 14, 1680]))

In [133]:
y_3276 = tf.pad(y, [[0,0], [0,0], [0,0], [0,0], [0,3276 - y.shape[-1]]], "REFLECT")

In [77]:
ind_0 = tf.cast(simulator.Resource_Grid_Mapper._data_ind, dtype=tf.int32)
# Determine the number of rows in ind_0
num_rows = tf.shape(ind_0)[0]

# Create indices to target all rows in the second column (index 1)
row_indices = tf.range(num_rows)
column_index = tf.constant(1, shape=[num_rows])
indices = tf.stack([row_indices, column_index], axis=1)

# Create the updates tensor with the desired values
updates = tf.ones([num_rows], dtype=ind_0.dtype)

# Apply the tensor scatter update
ind_1 = tf.tensor_scatter_nd_update(ind_0, indices, updates)

In [78]:
ind = tf.concat([ind_0, ind_1], axis = 0)
ind = tf.gather(ind, [0, 2, 3, 1], axis=1)
ind

<tf.Tensor: shape=(36960, 4), dtype=int32, numpy=
array([[   0,    0,    0,    0],
       [   0,    0,    1,    0],
       [   0,    0,    2,    0],
       ...,
       [   0,   13, 1677,    1],
       [   0,   13, 1678,    1],
       [   0,   13, 1679,    1]], dtype=int32)>

In [79]:
def predict(y):
    
    # Concat Real and Image of y
    y = tf.concat([tf.math.real(y), tf.math.imag(y)], axis=2)
    y = y[:,0]
    y = tf.transpose(y, perm=[0,3,2,1])

    pred = _model(y)
    print(pred.shape)
    pred = tf.transpose(pred, perm=[0,2,1,3])
    print(pred.shape,  pred[*ind[0]])
    return tf.gather_nd(pred, [[ind]])
    
    
pred = predict(y_3276)
pred

(1, 3276, 14, 2)
(1, 14, 3276, 2) tf.Tensor(7.867284, shape=(), dtype=float32)


<tf.Tensor: shape=(1, 1, 36960), dtype=float32, numpy=
array([[[  7.867284 ,  -3.770826 , -11.421184 , ...,  -8.5591545,
          -1.451167 , -10.918466 ]]], dtype=float32)>

In [80]:
def loss_cal(pred, labels):
  bce = tf.nn.sigmoid_cross_entropy_with_logits(labels, pred)
  bce = tf.reduce_mean(bce)
  # rate = tf.constant(1.0, tf.float32) - bce/tf.math.log(2.)
  # loss = -rate
  loss = bce
  return loss

loss_cal(pred, c)

<tf.Tensor: shape=(), dtype=float32, numpy=2.985408>

In [225]:
sysCfg = SystemConfig(
    NCellId = 0,
    FrequencyRange = 1,
    BandWidth = 100,
    Numerology = 1,
    CpType = 0,
    NTxAnt = 1,
    NRxAnt = 8,
    BwpNRb = 273,
    BwpRbOffset = 0,
    harqProcFlag = 0,
    nHarqProc = 1,
    rvSeq = 0
)
ueCfg = UeConfig(
    rvIdx = 0,
    TransformPrecoding = 0,
    Rnti = 24241,
    nId = 0,
    CodeBookBased = 0,
    DmrsPortSetIdx = [0],
    NLayers = 1,
    NumDmrsCdmGroupsWithoutData = 2,
    Tpmi = 0,
    FirstSymb = 0,
    NPuschSymbAll = 14,
    RaType = 1,
    FirstPrb = 0,
    NPrb = 100,
    FrequencyHoppingMode = 0,
    McsTable = 0,
    Mcs = 3,
    ILbrm = 0,
    nScId = 0,
    NnScIdId = 0,
    DmrsConfigurationType = 0,
    DmrsDuration = 1,
    DmrsAdditionalPosition = 1,
    PuschMappingType = 0,
    DmrsTypeAPosition = 2,
    HoppingMode = 0,
    NRsId = 0,
    Ptrs = 0,
    ScalingFactor = 0,
    OAck = 0,
    IHarqAckOffset = 11,
    OCsi1 = 0,
    ICsi1Offset = 7,
    OCsi2 = 0,
    ICsi2Offset = 0,
    NPrbOh = 0,
    nCw = 1,
    TpPi2Bpsk = 0
)
MyCfg = MyConfig(sysCfg, [ueCfg])
MyPuschCfg = MyPUSCHConfig(MyCfg)
MyPuschCfg.show()

Carrier Configuration
cyclic_prefix : normal
cyclic_prefix_length : 2.3437500000000002e-06
frame_duration : 0.01
frame_number : 0
kappa : 64.0
mu : 1
n_cell_id : 0
n_size_grid : 273
n_start_grid : 0
num_slots_per_frame : 20
num_slots_per_subframe : 2
num_symbols_per_slot : 14
slot_number : 4
sub_frame_duration : 0.001
subcarrier_spacing : 30
t_c : 5.086263020833334e-10
t_s : 3.2552083333333335e-08

PUSCH Configuration
My_Config : MyConfig(Sys=SystemConfig(NCellId=0, FrequencyRange=1, BandWidth=100, Numerology=1, CpType=0, NTxAnt=1, NRxAnt=8, BwpNRb=273, BwpRbOffset=0, harqProcFlag=0, nHarqProc=1, rvSeq=0), Ue=[UeConfig(rvIdx=0, TransformPrecoding=0, Rnti=24241, nId=0, CodeBookBased=0, DmrsPortSetIdx=[0], NLayers=1, NumDmrsCdmGroupsWithoutData=2, Tpmi=0, FirstSymb=0, NPuschSymbAll=14, RaType=1, FirstPrb=0, NPrb=100, FrequencyHoppingMode=0, McsTable=0, Mcs=3, ILbrm=0, nScId=0, NnScIdId=0, DmrsConfigurationType=0, DmrsDuration=1, DmrsAdditionalPosition=1, PuschMappingType=0, DmrsTypeAPosi

In [226]:
mySim = MySimulator(MyPuschCfg)

channel = OFDMChannel(channel_model=channel_model, resource_grid=mySim.resource_grid, 
                                    add_awgn=False, normalize_channel=True, return_channel=True)

b, c, y = mySim.sim(8, channel, 0.1)

In [227]:
tf.math.reduce_sum(y, [0,1,2,4])

<tf.Tensor: shape=(14,), dtype=complex64, numpy=
array([ -375.17508+837.1025j ,  -129.9725 -305.39713j,
        -493.94766-213.67807j, -1155.9622 -234.03107j,
         -15.678  -577.62744j,  -721.6193 +865.2868j ,
         722.3882 -473.68848j,  -179.3668 +211.04643j,
        -717.07196-485.4806j ,   477.3501 -405.5616j ,
         655.75824+118.03338j,   934.65094-916.54987j,
        -430.2192 -958.45636j,  -836.83826-670.0522j ], dtype=complex64)>

In [241]:
y_reposition = tf.gather(y, indices=[0,1,11,3,4,5,6,7,8,9,10,2,12,13], axis=-2)
y_reposition

<tf.Tensor: shape=(8, 1, 8, 14, 1200), dtype=complex64, numpy=
array([[[[[ 4.56030965e-01-4.95665550e-01j,
           -5.89814901e-01+1.36007905e+00j,
            8.05373073e-01+8.73277962e-01j, ...,
            7.07760274e-01-4.35498178e-01j,
            7.22862959e-01-2.18311444e-01j,
           -9.53727603e-01-5.65880537e-02j],
          [ 5.15272379e-01+2.42580503e-01j,
            9.53558564e-01-5.26518404e-01j,
            5.09117424e-01-4.47425663e-01j, ...,
            5.78299403e-01+2.51421243e-01j,
           -5.12282729e-01-1.08000445e+00j,
            2.37104580e-01+7.78426945e-01j],
          [-1.21389747e+00-8.48411679e-01j,
           -8.53024609e-03-4.64268982e-01j,
           -9.60168123e-01-1.14161873e+00j, ...,
            3.99939537e-01-2.36329183e-01j,
           -8.56692910e-01+1.80399269e-01j,
            3.13141674e-01-9.96033400e-02j],
          ...,
          [ 9.75992262e-01+8.30864727e-01j,
           -3.76347732e-03-2.39719436e-01j,
            8.06539774e-

In [242]:
tf.math.reduce_sum(y_reposition, [0,1,2,4])

<tf.Tensor: shape=(14,), dtype=complex64, numpy=
array([ -375.17508+837.1025j ,  -129.9725 -305.39713j,
         934.65094-916.54987j, -1155.9622 -234.03107j,
         -15.678  -577.62744j,  -721.6193 +865.2868j ,
         722.3882 -473.68848j,  -179.3668 +211.04643j,
        -717.07196-485.4806j ,   477.3501 -405.5616j ,
         655.75824+118.03338j,  -493.94766-213.67807j,
        -430.2192 -958.45636j,  -836.83826-670.0522j ], dtype=complex64)>

In [243]:
y_3276 = tf.tile(y_reposition, [1,1,1,1,(3276/y.shape[-1]).__ceil__()])
y_3276 = y_3276[...,:3276]
y_3276.shape

TensorShape([8, 1, 8, 14, 3276])

In [244]:
_y = y_3276[:,0]
_y = tf.concat([tf.math.real(_y), tf.math.imag(_y)], axis=1)
_y = tf.transpose(_y, perm=[0,3,2,1])
pred = _model(_y)

In [245]:
_pred = tf.concat([pred[...,0:2,:], pred[...,3:11,:], pred[...,12:14,:]],axis=-2)
_pred = tf.transpose(_pred, perm=[0,2,1,3])
_pred.shape

TensorShape([8, 12, 3276, 2])

In [246]:
c_pred = _pred[:,:,:y.shape[-1], :]
c_pred = tf.reshape(c_pred, [8,1,-1])
loss_cal(c_pred, c)

<tf.Tensor: shape=(), dtype=float32, numpy=3.4524963>

In [247]:
b_hat, crc = mySim.TB_Decode(c_pred)
crc

<tf.Tensor: shape=(8, 1), dtype=bool, numpy=
array([[False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False]])>